In [ ]:
:< sm'_operators
:< rust_operators
:< unsupported

#### file_system (test)

In [ ]:
///- --test static --symbol

:< testing

## fsharp

### file_mode

In [ ]:
:! file_mode' = $'System.IO.FileMode'

:| file_mode =
    | ModeCreateNew
    | ModeCreate
    | ModeOpen
    | ModeOpenOrCreate
    | Truncate
    | Append

:= file_mode = \|
    | ModeCreateNew => $'System.IO.FileMode.CreateNew' : file_mode'
    | ModeCreate => $'System.IO.FileMode.Create' : file_mode'
    | ModeOpen => $'System.IO.FileMode.Open' : file_mode'
    | ModeOpenOrCreate => $'System.IO.FileMode.OpenOrCreate' : file_mode'
    | Truncate => $'System.IO.FileMode.Truncate' : file_mode'
    | Append => $'System.IO.FileMode.Append' : file_mode'

### file_access

In [ ]:
:! file_access' = $'System.IO.FileAccess'

:| file_access =
    | AccessRead
    | AccessWrite
    | AccessReadWrite

:= file_access = \|
    | AccessRead => $'System.IO.FileAccess.Read' : file_access'
    | AccessWrite => $'System.IO.FileAccess.ReadWrite' : file_access'
    | AccessReadWrite => $'System.IO.FileAccess.ReadWrite' : file_access'

### file_share

In [ ]:
:! file_share' = $'System.IO.FileShare'

:| file_share =
    | ShareNone
    | ShareRead
    | ShareWrite
    | ShareReadWrite
    | ShareDelete

:= file_share = \|
    | ShareNone => $'System.IO.FileShare.None' : file_share'
    | ShareRead => $'System.IO.FileShare.Read' : file_share'
    | ShareWrite => $'System.IO.FileShare.Write' : file_share'
    | ShareReadWrite => $'System.IO.FileShare.ReadWrite' : file_share'
    | ShareDelete => $'System.IO.FileShare.Delete' : file_share'

:! file_info =
    `(
        backend_switch `(()) `({}) {
            Fsharp = (\ () =>
                global """
                    type System_IO_FileInfo = System.IO.FileInfo

                    """
                        ) : () -> ()
        }
        $'' : $'System_IO_FileInfo'
    )

:= file_info (path : string) : file_info =
    run_target \|
        | Fsharp => \ () => $'!path |> `file_info '
        | _ => \ () => null ()

:! directory_info =
    `(
        backend_switch `(()) `({}) {
            Fsharp = (\ () =>
                global """
                    type System_IO_DirectoryInfo = System.IO.DirectoryInfo

                    """
                        ) : () -> ()
        }
        $'' : $'System_IO_DirectoryInfo'
    )

:= directory_info (path : string) : directory_info =
    run_target \|
        | Fsharp => \ () => $'!path |> `directory_info '
        | _ => \ () => null ()

:= directory_info_name (info : directory_info) : string =
    run_target \|
        | Fsharp => \ () => info |> $'_.Name'
        | _ => \ () => null ()

:= directory_info_full_name (info : directory_info) : string =
    run_target \|
        | Fsharp => \ () => info |> $'_.FullName'
        | _ => \ () => null ()

:! file_attributes = $'System.IO.FileAttributes'

:~ directory_info_attributes (info : directory_info) : file_attributes =
    run_target \|
        | Fsharp => \ () => info |> $'_.Attributes'
        | _ => \ () => null ()

:~ file_attributes_reparse_point () : file_attributes =
    run_target \|
        | Fsharp => \ () => $'`file_attributes.ReparsePoint'
        | _ => \ () => null ()

:~ file_attributes_has_flag (flag : file_attributes) (file_attributes : file_attributes) : bool =
    run_target \|
        | Fsharp => \ () => $'!file_attributes.HasFlag !flag '
        | _ => \ () => null ()

:~ create_directory (path : string) : directory_info =
    run_target \|
        | Fsharp => \ () => path |> $'System.IO.Directory.CreateDirectory'
        | _ => \ () => null ()

:~ directory_get_files (path : string) : array_base string =
    run_target \|
        | Fsharp => \ () => path |> $'System.IO.Directory.GetFiles'
        | _ => \ () => null ()

:~ write_all_text_async (path : string) (text : string) : _ () =
    run_target \|
        | Fsharp => \ () => $'System.IO.File.WriteAllTextAsync (!path, !text)' |> async::await_task
        | _ => \ () => null ()

:! file_system_info = $'System.IO.FileSystemInfo'

:= get_source_directory () =
    run_target \|
        | Rust => \ () => $'Rc::<str>::from(env\!("CARGO_MANIFEST_DIR"))' : string
        | _ => \ () => $'__SOURCE_DIRECTORY__' : string

#### get_source_directory (test)

In [1]:
///- --test --symbol
///> fsharp

get_source_directory ()
|> directory_info
|> directory_info_name
|> _assert_in [ "spiral"; "fsharp" ]

17:42:17 1d _assert_in / { x = spiral; x' = spiral }

## rust

In [ ]:
:! display =
    `(
        rust::rust_type
            `($'std_path_Display')
            `($'std::path::Display')
            "type std_path_Display = string"
    )

:! path =
    `(
        rust::rust_type
            `($'std_path_Path')
            `($'std::path::Path')
            "type std_path_Path = class end"
    )

:! path_buf =
    `(
        rust::rust_type
            `($'std_path_PathBuf')
            `($'std::path::PathBuf')
            "type std_path_PathBuf = string"
    )

:= new_path_buf (path : sm'::std_string) : path_buf =
    run_target \|
        | Rust => \ () => !\\(path, $'"std::path::PathBuf::from($0)"')
        | _ => \ () => path |> convert::unbox

:= path_buf_join (s : string) (path_buf : path_buf) : path_buf =
    !\\((path_buf, s |> sm'::to_std_string), $'"$0.join($1)"')

:= path_buf_display (path_buf : path_buf) : display =
    run_target_args (\ () => path_buf) \|
        | Rust => \ path_buf =>
            := path_buf = ~~ path_buf
            !\($'"!path_buf.display()"')
        | _ => \ path_buf => path_buf |> convert::unbox

:= path_buf_file_name (path : path_buf) : optionm'::option' (rust::ref sm'::os_str) =
    !\\(path, $'"$0.file_name()"')

:= path_buf_exists (path_buf : path_buf) : bool =
    !\\(path_buf, $'"$0.exists()"')

:= path_buf_is_dir (path_buf : path_buf) : bool =
    !\\(path_buf, $'"$0.is_dir()"')

:= path_buf_is_file (path_buf : path_buf) : bool =
    !\\(path_buf, $'"$0.is_file()"')

:= path_buf_parent (path_buf : path_buf) : optionm'::option' path_buf =
    !\\(path_buf, $'"$0.parent().map(std::path::PathBuf::from)"')

:! dir_entry =
    `(
        rust::rust_type
            `($'async_walkdir_DirEntry')
            `($'async_walkdir::DirEntry')
            "type async_walkdir_DirEntry = class end"
    )

:! walk_dir =
    `(
        rust::rust_type
            `($'async_walkdir_WalkDir')
            `($'async_walkdir::WalkDir')
            "type async_walkdir_WalkDir = class end"
    )

:! async_walkdir_filtering =
    `(
        rust::rust_type
            `($'async_walkdir_Filtering')
            `($'async_walkdir::Filtering')
            "type async_walkdir_Filtering = class end"
    )

:| filtering =
    | Ignore
    | IgnoreDir
    | Continue

:! async_walkdir_error =
    `(
        rust::rust_type
            `($'async_walkdir_Error')
            `($'async_walkdir::Error')
            "type async_walkdir_Error = class end"
    )

:= new_walk_dir (dir : string) : walk_dir =
    !\\(dir, $'"async_walkdir::WalkDir::new(&*$0)"')

:= walk_dir_filter (fn : dir_entry -> async::future_pin_send filtering) (walk_dir : walk_dir) : walk_dir =
    ? is_rust () ?>
        global """
            struct SpiralAssertSend<T>(T);
            unsafe impl<T> Send for SpiralAssertSend<T> {}
            impl<T: std::future::Future + Unpin> std::future::Future for SpiralAssertSend<T> {
                type Output = T::Output;
                fn poll(mut self: std::pin::Pin<&mut Self>, cx: &mut std::task::Context<'_>) -> std::task::Poll<T::Output> { std::pin::Pin::new(&mut self.0).poll(cx) }
            }
            """
        := fn entry =
            := future = fn entry
            async::new_future_move_send \ () =>
                := result = future |> async::await_send
                := filtering : async_walkdir_filtering =
                    ?= result ?|
                    | Ignore => !\($'"async_walkdir::Filtering::Ignore"')
                    | IgnoreDir => !\($'"async_walkdir::Filtering::IgnoreDir"')
                    | Continue => !\($'"async_walkdir::Filtering::Continue"')
                filtering
        !\\((walk_dir, fn), $'"{ let f = SpiralAssertSend($1); async_walkdir::WalkDir::filter($0, move |x| { let f = &f; SpiralAssertSend((f.0)(x)) }) }"')
    |:
        := fn entry = async::new_future_send \ () =>
            := result = fn entry |> async::await_send
            := filtering : async_walkdir_filtering =
                ?= result ?|
                | Ignore => !\($'"async_walkdir::Filtering::Ignore"')
                | IgnoreDir => !\($'"async_walkdir::Filtering::IgnoreDir"')
                | Continue => !\($'"async_walkdir::Filtering::Continue"')
            filtering
        !\\((walk_dir, fn), $'"async_walkdir::WalkDir::filter($0, move |x| $1(x))"')

:! file_type =
    `(
        rust::rust_type
            `($'std_fs_FileType')
            `($'std::fs::FileType')
            "type std_fs_FileType = class end"
    )

:= dir_entry_file_type (dir_entry : dir_entry) : async::future_pin_send (resultm::result' file_type stream::io_error) =
    := dir_entry = dir_entry |> rust::emit
    !\($'"Box::pin(async_walkdir::DirEntry::file_type(&!dir_entry))"')

:= file_type_is_dir (file_type : file_type) : bool =
    !\\(file_type, $'"std::fs::FileType::is_dir(&$0)"')

:! file =
    `(
        rust::rust_type
            `($'std_fs_File')
            `($'std::fs::File')
            "type std_fs_File = class end"
    )

:= file_open (path : string) : resultm::result' file stream::io_error =
    !\($'"std::fs::File::open(&*!path)"')

:= rename (to : string) (path : string) : resultm::result' () stream::io_error =
    !\($'"std::fs::rename(&*!path, &*!to)"')

:= dir_entry_path (dir_entry : dir_entry) : path_buf =
    !\\(dir_entry, $'"async_walkdir::DirEntry::path(&$0)"')

:= create_dir_all (path : string) : resultm::result' () stream::io_error =
    !\\(path, $'"std::fs::create_dir_all(&*$0)"')

:= file_info_link_target (file_info : file_info) : string =
    run_target \|
        | Fsharp => \ () =>
            file_info |> $'_.LinkTarget'
        | _ => \ () => null ()

:= read (path : string) : resultm::result' (am'::vec u8) stream::io_error =
    !\\(path, $'"std::fs::read(&*$0)"')

## typescript

In [ ]:
:= ts_path_join (b : string) (a : string) : string =
    global "import * as spiral_node_path from 'node:path';"
    $'spiral_node_path.join(!a, !b)' : string

## file_system

### (< />)

In [ ]:
:~ (</>) (a : string) (b : string) : string =
    run_target_args (\ () => a, b) \|
        | Rust => \ a, b =>
            a
            |> sm'::to_std_string
            |> new_path_buf
            |> path_buf_join b
            |> path_buf_display
            |> sm'::format'
            |> sm'::from_std_string
        | TypeScript => \ a, b =>
            a |> ts_path_join b
        | Fsharp => \ a, b =>
            $'System.IO.Path.Combine (!a, !b)'
        | _ => \ a, b =>
            ? a = "" ?> b
            |? (a |> sm'::ends_with "/") || (a |> sm'::ends_with "\\") ?> a ++# b
            |: a ++# "/" ++# b

#### path_join_without_a_native_arm (test)

In [2]:
///- --test --symbol
///> gleam
///> zig
///> lua

"dir" </> "file" |> _assert_eq "dir/file"
"dir/" </> "file" |> _assert_eq "dir/file"
"" </> "file" |> _assert_eq "file"

.gleam (gleam)
__assert_eq { expected = dir/file }
__assert_eq { expected = dir/file }
__assert_eq { expected = file }
.zig (zig)
__assert_eq { expected = dir/file }
__assert_eq { expected = dir/file }
__assert_eq { expected = file }
.lua (lua)
__assert_eq { expected = dir/file }
__assert_eq { expected = dir/file }
__assert_eq { expected = file }

### get_temp_path

In [ ]:
:= without_trailing_separator (path : string) : string =
    ? sm'::length path > 1i32 && (sm'::ends_with "/" path || sm'::ends_with "\\" path)
    ?> sm'::substring 0i32 (sm'::length path - 1i32) path
    |: path

:! gleam_dynamic = $'Dynamic'

:! gleam_atom = $'atom.Atom'

:! gleam_charlist = $'charlist.Charlist'

:= file_system_gleam_env () =
    global "import gleam/dynamic.{type Dynamic}"
    global "import gleam/erlang/charlist"
    global """
        @external(erlang, "os", "getenv")
        fn spiral_fs_os_getenv(name: charlist.Charlist) -> Dynamic

        @external(erlang, "gleam_stdlib", "identity")
        fn spiral_fs_env_coerce(value: a) -> b

        """

:= gleam_env_value (name : string) : gleam_dynamic =
    file_system_gleam_env ()
    := chars : gleam_charlist = $'charlist.from_string(!name)'
    $'spiral_fs_os_getenv(!chars)'

:= gleam_env_unset (value : gleam_dynamic) : bool =
    := unset : gleam_dynamic = $'spiral_fs_env_coerce(False)'
    $'!value == !unset'

:= gleam_env_text (value : gleam_dynamic) : string =
    := chars : gleam_charlist = $'spiral_fs_env_coerce(!value)'
    $'charlist.to_string(!chars)'

:~ ^ gleam_temp_directory_from (names : list string) : string =
    ?= names ?|
    | [] => "/tmp"
    | name :: rest =>
        := value = gleam_env_value name
        ? gleam_env_unset value ?> gleam_temp_directory_from rest
        |:
            := dir = gleam_env_text value
            ? dir = "" ?> gleam_temp_directory_from rest |: dir

:= env_temp_dir () : string =
    := tmpdir = env::get_environment_variable "TMPDIR"
    ? tmpdir = "" ?>
        := temp = env::get_environment_variable "TEMP"
        ? temp = "" ?> "/tmp" |: temp
    |: tmpdir

:~ get_temp_path () : string =
    run_target \|
        | Rust => \ () =>
            !\($'"std::env::temp_dir()"')
            |> path_buf_display
            |> sm'::format'
            |> sm'::from_std_string
            |> without_trailing_separator
        | Fsharp => \ () =>
            $'System.IO.Path.GetTempPath' () |> without_trailing_separator
        | Gleam => \ () =>
            file_system_gleam_env ()
            gleam_temp_directory_from [ "TMPDIR"; "TEMP"; "TMP" ] |> without_trailing_separator
        | C => \ () => env_temp_dir ()
        | Zig => \ () => env_temp_dir ()
        | target => \ () => ~~ "file_system.get_temp_path / target: " ++#? target |> failwith

#### get_temp_path_is_set_on_native_targets (test)

In [3]:
///- --test --symbol
///> c
///> zig

(sm'::length (get_temp_path ()) > 0i32) |> _assert_eq !+

.c (c)
__assert_eq { expected = true }
.zig (zig)
__assert_eq { expected = true }

In [ ]:
:= file_system_gleam_paths () =
    global "import gleam/dynamic.{type Dynamic}"
    global "import gleam/erlang/atom"
    global "import gleam/string"
    global "import gleam/list"
    global """
        @external(erlang, "filelib", "is_dir")
        fn spiral_fs_is_dir(path: String) -> Bool

        @external(erlang, "file", "delete")
        fn spiral_fs_delete(path: String) -> Dynamic

        @external(erlang, "filename", "basename")
        fn spiral_fs_basename(path: String) -> String

        @external(erlang, "file", "write_file")
        fn spiral_fs_write_file_modes(path: String, data: String, modes: List(atom.Atom)) -> Dynamic

        @external(erlang, "gleam_stdlib", "identity")
        fn spiral_fs_path_coerce(value: a) -> b

        @external(erlang, "filename", "extension")
        fn spiral_fs_extension(path: String) -> String

        @external(erlang, "filename", "absname")
        fn spiral_fs_absname(path: String) -> String
        """



:= gleam_atom_named (name : string) : gleam_atom = $'atom.create(!name)'

:= gleam_atoms_of (a : gleam_atom) : listm'::list' gleam_atom = $'[!a]'

:= gleam_atom_of_dynamic (d : gleam_dynamic) : gleam_atom = $'spiral_fs_path_coerce(!d)'

:= gleam_atoms_equal (a : gleam_atom) (b : gleam_atom) : bool = $'!a == !b'

:= gleam_written (path : string) (text : string) (modes : listm'::list' gleam_atom) : gleam_dynamic =
    $'spiral_fs_write_file_modes(!path, !text, !modes)'

:= gleam_segment_step (acc : list string) (segment : string) : list string =
    ? segment = "" || segment = "." ?> acc
    |? segment = ".." ?>
        ?= acc ?|
        | _ :: up => up
        | [] => []
    |: (segment :: acc)

:= gleam_lexical_full_path (path : string) : string =
    file_system_gleam_paths ()
    := absolute : string = $'spiral_fs_absname(!path)'
    ?= sm'::split_list "/" absolute ?|
    | root :: segments =>
        root ++# "/" ++# (segments |> listm::fold gleam_segment_step [] |> listm::rev |> sm'::concat_list "/")
    | [] => path

:= gleam_dotted_extension (path : string) : string =
    file_system_gleam_paths ()
    := base : string = $'spiral_fs_basename(!path)'
    := rest = ? sm'::starts_with "." base ?> sm'::substring 1i32 (sm'::length base - 1i32) base |: base
    $'spiral_fs_extension(!rest)' : string

:= gleam_append_text (path : string) (text : string) : () =
    file_system_gleam_paths ()
    := written = gleam_written path text (gleam_atoms_of (gleam_atom_named "append")) |> gleam_atom_of_dynamic
    ? gleam_atoms_equal written (gleam_atom_named "ok") |> not
    ?> failwith "file_system.append_all_text: the append failed"

:~ get_file_name (path : string) : string =
    run_target_args' path \|
        | Fsharp => \ path =>
            path |> $'System.IO.Path.GetFileName'
        | Rust => \ path =>
            path
            |> sm'::to_std_string
            |> new_path_buf
            |> path_buf_file_name
            |> optionm'::map' sm'::from_os_str_ref
            |> optionm'::unbox
            |> optionm'::default_value ""
        | TypeScript => \ path =>
            global "import * as spiral_node_path from 'node:path';"
            $'spiral_node_path.basename(!path)' : string
        | Gleam => \ path =>
            file_system_gleam_paths ()
            $'spiral_fs_basename(!path)' : string
        | target =>
            \ path => ~~ "file_system.get_file_name / target: " ++#? target ++# " / path: " ++# path |> failwith

:~ get_file_name_without_extension (path : string) : string =
    run_target_args' path \|
        | Rust => \ path =>
            := path_buf = path |> sm'::to_std_string |> new_path_buf
            := file_stem = !\\(path_buf, $'"$0.file_stem()"')
            ?= file_stem |> optionm'::map' sm'::from_os_str_ref |> optionm'::unbox ?|
            | Some file_stem => file_stem
            | None => ""
        | _ => \ path =>
            path |> $'System.IO.Path.GetFileNameWithoutExtension'

:~ get_extension (path : string) : string =
    run_target_args' path \|
        | Rust => \ path =>
            := path_buf = path |> sm'::to_std_string |> new_path_buf
            := extension = !\\(path_buf, $'"$0.extension()"')
            ?= extension |> optionm'::map' sm'::from_os_str_ref |> optionm'::unbox ?|
            | Some extension => "." ++# extension
            | None => ""
        | Gleam => \ path =>
            file_system_gleam_paths ()
            gleam_dotted_extension path
        | Fsharp => \ path =>
            $'(let name = System.IO.Path.GetFileName (!path : string) in if name.StartsWith "." && name.IndexOf(\
                ".", 1, System.StringComparison.Ordinal) < 0 then "" else System.IO.Path.GetExtension (!path : string))'
            : string
        | _ => \ path =>
            path |> $'System.IO.Path.GetExtension'

#### get_extension_keeps_the_dot_and_skips_dotfiles (test)

In [4]:
///- --test --symbol
///> rust
///> gleam

(get_extension "a/b.txt", get_extension "a/b", get_extension "a/.bashrc", get_extension "a.tar.gz")
|> _assert_eq (".txt", "", "", ".gz")

.rs (rust)
__assert_eq { expected = (.txt, , , .gz) }
.gleam (gleam)
__assert_eq { expected = (.txt, , , .gz) }

0

In [ ]:
:~ directory_separator_char () : char =
    run_target \|
        | Rust => \ () => $'(std::path::MAIN_SEPARATOR as u8)' : char
        | TypeScript _ => \ () => '/'
        | Zig => \ () => $'std.fs.path.sep' : char
        | _ => \ () => $'System.IO.Path.DirectorySeparatorChar'

### get_current_directory

In [ ]:
:= file_system_gleam_stat () =
    global "import gleam/dynamic.{type Dynamic}"
    global "import gleam/erlang/atom"
    global "import gleam/erlang/charlist"
    global """
        @external(erlang, "file", "get_cwd")
        fn spiral_fs_get_cwd() -> Result(charlist.Charlist, Dynamic)

        @external(erlang, "file", "read_file_info")
        fn spiral_fs_read_file_info(path: String, options: List(#(atom.Atom, atom.Atom))) -> Result(Dynamic, Dynamic)

        @external(erlang, "erlang", "element")
        fn spiral_fs_stat_element(index: Int, tuple: Dynamic) -> Int

        """

:= gleam_cwd () : string =
    file_system_gleam_stat ()
    ?= ($'spiral_fs_get_cwd()' : resultm::result' gleam_charlist gleam_dynamic) |> resultm::unbox ?|
    | Ok chars => $'charlist.to_string(!chars)' : string
    | Error _ => failwith "file_system.get_current_directory: file:get_cwd failed"

:= gleam_mtime_ms (path : string) : u64 =
    file_system_gleam_stat ()
    := time = gleam_atom_named "time"
    := posix = gleam_atom_named "posix"
    := option : gleam_dynamic = $'##(!time, !posix)'
    := options : listm'::list' gleam_dynamic = $'[!option]'
    ?= ($'spiral_fs_read_file_info(!path, !options)' : resultm::result' gleam_dynamic gleam_dynamic) |> resultm::unbox ?|
    | Ok info => ($'spiral_fs_stat_element(6, !info)' : u64) * 1000u64
    | Error _ => 0u64

:~ get_current_directory () : string =
    run_target \|
        | Rust => \ () =>
            := current_dir = !\($'"std::env::current_dir()"') : resultm::result' path_buf stream::io_error
            current_dir
            |> resultm::unwrap'
            |> path_buf_display
            |> sm'::format'
            |> sm'::from_std_string
        | Fsharp => \ () =>
            $'System.IO.Directory.GetCurrentDirectory' ()
        | TypeScript _ => \ () =>
            global "import * as spiral_node_fs from 'node:fs';"
            $'spiral_node_fs.realpathSync(".")' : string
        | Zig => \ () =>
            global "var spiral_io_threaded: std.Io.Threaded = .init_single_threaded;"
            $'(std.process.currentPathAlloc(spiral_io_threaded.io(), spiral_gpa) catch \
                \@panic("file_system.get_current_directory failed"))' : string
        | Gleam => \ () =>
            file_system_gleam_stat ()
            gleam_cwd ()
        | Pascal => \ () => $'GetCurrentDir()' : string
        | _ => \ () => null ()

In [5]:
///- --test --symbol

get_current_directory ()
|> _assert_contains (directory_separator_char ())

__assert_contains { expected = / }

In [ ]:
:~ directory_exists (path : string) : bool =
    run_target_args' path \|
        | Zig => \ _ =>
            global "var spiral_io_threaded: std.Io.Threaded = .init_single_threaded;"
            $'(if (std.Io.Dir.cwd().statFile(spiral_io_threaded.io(), !path, .{})) |s| s.kind == .directory else |_| \
                false)' : bool
        | Fsharp => \ path =>
            path |> $'System.IO.Directory.Exists'
        | Rust => \ path =>
            := path = path |> sm'::to_std_string |> new_path_buf
            path_buf_exists path && path_buf_is_dir path
        | TypeScript => \ path =>
            global "import * as spiral_node_fs from 'node:fs';"
            $'(spiral_node_fs.existsSync(!path) && spiral_node_fs.statSync(!path).isDirectory())' : bool
        | Gleam => \ path =>
            file_system_gleam_paths ()
            $'spiral_fs_is_dir(!path)' : bool
        | Pascal => \ path => $'DirectoryExists(!path)' : bool
        | _ => \ _ => null ()

:~ directory_get_parent (path : string) : optionm'::option' string =
    run_target_args' path \|
        | Fsharp => \ path =>
            := parent : directory_info = path |> $'System.IO.Directory.GetParent'
            ? parent =. null ()
            ?> None
            |: parent |> directory_info_full_name |> Some
            |> optionm'::box
        | Rust => \ path =>
            := path_buf = path |> sm'::to_std_string |> new_path_buf
            := parent = path_buf |> path_buf_parent
            parent
            |> optionm'::map' (path_buf_display >> sm'::format' >> sm'::from_std_string)
        | TypeScript _ => \ path =>
            global "import * as spiral_node_path from 'node:path';"
            := parent = $'spiral_node_path.dirname(!path)' : string
            ? parent = path
            ?> None
            |: parent |> Some
            |> optionm'::box
        | _ => \ _ => optionm'::none' ()

:= directory_parent (path : string) : option string =
    run_target_args' path \|
        | Pascal => \ path =>
            := parent = $'ExtractFileDir(ExcludeTrailingPathDelimiter(!path))' : string
            ? parent = path || parent = "" ?> None |: Some parent
        | _ => \ path => path |> directory_get_parent |> optionm'::unbox

:= directory_create_all (path : string) : result () string =
    run_target_args' path \|
        | Pascal => \ path =>
            ? $'ForceDirectories(!path)' : bool ?> Ok () |: Error ("directory_create_all / cannot create " ++# path)
        | _ => \ path => path |> create_dir_all |> resultm::map_error' sm'::format |> resultm::unbox

#### directory_create_all_makes_nested_directories (test)

In [6]:
///- --test --symbol
///> rust
///> pascal

:= path = get_current_directory () </> "directory_create_all_test" </> "a" </> "b"
?= directory_create_all path ?|
| Ok () => ? directory_exists path = !- ?> failwith "the nested directory exists after directory_create_all"
| Error message => failwith message

pascal: unavailable on this machine (`dcc32` was not found (nor fpc))

0

In [ ]:
:~ create_temp_path' (guid : guid::guid) =
    run_target_args' guid \|
        | _ => \ guid =>
            get_temp_path ()
            </> ~~ "!create_temp_path_"
            </> (env::get_entry_assembly_name ())
            </> (guid |> sm'::obj_to_string)

#### create_temp_path (test)

In [7]:
///- --test --symbol
///> rust -d chrono

guid::hash_guid ""
|> create_temp_path'
|> _assert_contains (directory_separator_char ())

__assert_contains { expected = / }

In [ ]:
:~ create_temp_path () =
    run_target \|
        | _ => \ () =>
            date_time::now ()
            |> date_time::new_guid_from_date_time
            |> create_temp_path'

:~ file_copy (new_path : string) (old_path : string) : () =
    run_target_args' (old_path, new_path) \|
        | Fsharp => \ old_path, new_path =>
            $'System.IO.File.Copy (!old_path, !new_path, true)'
        | Rust => \ old_path, new_path =>
            := result : _ _ stream::io_error = !\\((old_path, new_path), $'"std::fs::copy(&*$0, &*$1)"')
            ?= result |> resultm::map_error' sm'::format' |> resultm::unbox ?|
            | Ok (result : u64) =>
                trace Debug
                    \ () => "file_system.file_copy"
                    \ () => { old_path new_path result }
            | Error error =>
                trace Warning
                    \ () => "file_system.file_copy"
                    \ () => { old_path new_path error }
        | _ => \ _ => ()

:= file_system_gleam_files () =
    global "import gleam/bit_array"
    global "import gleam/dynamic.{type Dynamic}"
    global "import gleam/erlang/atom"
    global """
        @external(erlang, "file", "read_file")
        fn spiral_fs_read_file(path: String) -> Result(BitArray, Dynamic)

        @external(erlang, "file", "write_file")
        fn spiral_fs_write_file(path: String, data: String) -> Dynamic

        @external(erlang, "filelib", "is_regular")
        fn spiral_fs_is_regular(path: String) -> Bool

        @external(erlang, "gleam_stdlib", "identity")
        fn spiral_fs_coerce(value: a) -> b

        """

:! gleam_bytes = $'BitArray'

:= gleam_read_text (path : string) : string =
    file_system_gleam_files ()
    ?= ($'spiral_fs_read_file(!path)' : resultm::result' gleam_bytes gleam_dynamic) |> resultm::unbox ?|
    | Ok bytes =>
        ?= ($'bit_array.to_string(!bytes)' : resultm::result' string ()) |> resultm::unbox ?|
        | Ok text => text
        | Error _ => failwith "file_system.read_all_text: the file is not UTF-8"
    | Error _ => failwith "file_system.read_all_text: file:read_file failed"

:= gleam_write_text (path : string) (text : string) : () =
    file_system_gleam_files ()
    := raw : gleam_dynamic = $'spiral_fs_write_file(!path, !text)'
    := written : gleam_atom = $'spiral_fs_coerce(!raw)'
    ? gleam_atoms_equal written (gleam_atom_named "ok") |> not
    ?> failwith "file_system.write_all_text: the write failed"

:! lua_file = $'any'

:= lua_file_open (path : string) (mode : string) : lua_file = $'io.open(!path, !mode)' : lua_file

:= lua_file_present (file : lua_file) : bool = $'(!file) ~= nil' : bool

:= lua_file_write (file : lua_file) (text : string) : () = $'(!file):write(!text)' : ()

:= lua_file_read_all (file : lua_file) : string = $'(!file):read("*a")' : string

:= lua_file_close (file : lua_file) : () = $'(!file):close()' : ()

:= lua_file_remove (path : string) : () = $'os.remove(!path)' : ()

:~ file_exists (path : string) : bool =
    run_target_args' path \|
        | Zig => \ _ =>
            global "var spiral_io_threaded: std.Io.Threaded = .init_single_threaded;"
            $'(if (std.Io.Dir.cwd().statFile(spiral_io_threaded.io(), !path, .{})) |s| s.kind == .file else |_| false)' : bool
        | Fsharp => \ path =>
            path |> $'System.IO.File.Exists'
        | Rust => \ path =>
            := path_buf = path |> sm'::to_std_string |> new_path_buf
            path_buf_exists path_buf && path_buf_is_file path_buf
        | Gleam => \ path =>
            file_system_gleam_files ()
            $'spiral_fs_is_regular(!path)' : bool
        | TypeScript => \ path =>
            global "import * as spiral_node_fs from 'node:fs';"
            $'(spiral_node_fs.existsSync(!path) && spiral_node_fs.statSync(!path).isFile())' : bool
        | Gleam _ => \ path =>
            global """
                @external(erlang, "filelib", "is_regular")
                fn spiral_filelib_is_regular(path: String) -> Bool
                """
            $'spiral_filelib_is_regular(!path)' : bool
        | Lua => \ path =>
            := file = lua_file_open path "r"
            ? lua_file_present file ?>
                lua_file_close file
                !+
            |: !-
        | Pascal => \ path => $'FileExists(!path)' : bool
        | _ => \ _ => null ()

:= write_all_text (path : string) (text : string) : () =
    run_target_args' (path, text) \|
        | Zig => \ _ =>
            global "var spiral_io_threaded: std.Io.Threaded = .init_single_threaded;"
            $'std.Io.Dir.cwd().writeFile(spiral_io_threaded.io(), .{ .sub_path = !path, .data = !text }) catch \
                \@panic("file_system.write_all_text failed")' : ()
        | Fsharp => \ path, text =>
            $'System.IO.File.WriteAllText (!path, !text)'
        | Rust => \ path, text =>
            !\\((path, text), $'"std::fs::write(&*$0, &*$1).unwrap()"')
        | Gleam => \ path, text =>
            file_system_gleam_files ()
            gleam_write_text path text
        | Lua => \ path, text =>
            := file = lua_file_open path "w"
            lua_file_write file text
            lua_file_close file
        | Pascal => \ path, text =>
            := handle = $'FileCreate(!path)' : i64
            ? handle = -1i64 ?> failwith "file_system.write_all_text / cannot create the file"
            := length = sm'::length text
            ? length > 0i32 ?>
                := first = 1i32
                := written = $'FileWrite(!handle, (!text)[!first], !length)' : i32
                ? written <> length ?> failwith "file_system.write_all_text / short write"
            $'FileClose(!handle)' : ()
        | _ => \ _ => ()

:= read_all_bytes (path : string) : am'::vec u8 =
    run_target \|
        | Fsharp => \ () =>
            $'!path |> System.IO.File.ReadAllBytes'
            |> am'::to_vec
        | Rust => \ () =>
            path |> read |> resultm::unwrap'
        | _ => \ () => null ()

:= c_read_file_global () : () =
    global "#include <stdint.h>"
    global "#include <stdio.h>"
    global (
        "static long spiral_read_file(const char * path, uint8_t * dst, long cap) { FILE * f = fopen(path, \"rb\"); "
        ^^ "if (!f) return -1; if (!dst) { fseek(f, 0, SEEK_END); long n = ftell(f); fclose(f); return n; } "
        ^^ "long n = (long)fread(dst, 1, (size_t)cap, f); fclose(f); return n; }"
    )

:= c_read_bytes (path : string) : array u8 =
    c_read_file_global ()
    := size : i32 = $'(int32_t)spiral_read_file(!path->ptr, NULL, 0)' : i32
    ? size < 0 ?> failwith "file_system.read_bytes / cannot open the file"
    := bytes : array u8 = create size
    := read : i32 = $'(int32_t)spiral_read_file(!path->ptr, !bytes->ptr, !bytes->len)' : i32
    ? read <> size ?> failwith "file_system.read_bytes / short read"
    bytes

:= zig_io () : () = global "var spiral_io_threaded: std.Io.Threaded = .init_single_threaded;"

:= zig_read_bytes (path : string) : array u8 =
    zig_io ()
    $'(std.Io.Dir.cwd().readFileAlloc(spiral_io_threaded.io(), !path, spiral_gpa, .unlimited) catch \
        \@panic("file_system.read_bytes failed"))' : array u8

:= read_bytes (path : string) : array u8 =
    run_target \|
        | C => \ () => c_read_bytes path
        | Zig => \ () => zig_read_bytes path
        | _ => \ () => am'::from_vec (read_all_bytes path)

#### read_bytes_reads_a_file_on_c (test)

In [8]:
///- --test --symbol
///> c

:= ~path = get_temp_path () ++# "/spiral_read_bytes_c.bin"
$'{ FILE * f = fopen(!path->ptr, "wb"); fputs("AB", f); fclose(f); }' : ()
:= b = read_bytes path
(length b, index b 0, index b 1) |> _assert_eq (2i32, 65u8, 66u8)

__assert_eq { expected = (2, 65, 66) }

#### read_bytes_reads_a_written_text_file (test)

In [9]:
///- --test --symbol
///> rust
///> zig

:= ~path = get_temp_path () ++# "/spiral_read_bytes_text.bin"
write_all_text path "AB"
:= b = read_bytes path
(length b, index b 0, index b 1) |> _assert_eq (2i32, 65u8, 66u8)

.rs (rust)
__assert_eq { expected = (2, 65, 66) }
.zig (zig)
__assert_eq { expected = (2, 65, 66) }

0

In [ ]:
:= c_stdio () : () = global "#include <stdio.h>"

:= c_open_at () : () =
    c_stdio ()
    global (
        "static FILE * spiral_open_at(const char * p, long at) { FILE * f = fopen(p, \"rb\"); "
        ^^ "if (f && fseek(f, at, SEEK_SET)) { fclose(f); f = NULL; } return f; }"
    )

:= c_read_open () : () =
    c_stdio ()
    global (
        "static long spiral_read_open(FILE * f, void * d, long n) { "
        ^^ "long r = f ? (long)fread(d, 1, (size_t)n, f) : -1; if (f) fclose(f); return r; }"
    )

:= c_read_region_global () : () =
    c_open_at ()
    c_read_open ()
    global (
        "static long spiral_read_region(const char * p, long at, void * d, long n) { "
        ^^ "return spiral_read_open(spiral_open_at(p, at), d, n); }"
    )

:= c_read_region \* el. (path : string) (at : i32) (out : array el) (bytes : i32) : i32 =
    c_read_region_global ()
    $'(int32_t)spiral_read_region(!path->ptr, !at, !out->ptr, !bytes)' : i32

:= c_read_into \* el. (path : string) (at : i32) (out : array el) (size : i32) : array el =
    := bytes = size * length out
    ? c_read_region path at out bytes <> bytes ?> failwith "file_system.read_region / short read"
    out

:~ ^ copy_prefix_from (b : array u8) (out : array u8) (i : i32) : () =
    ? i < length out ?>
        set out i (index b i)
        copy_prefix_from b out (i + 1)

:= read_prefix (path : string) (count : i32) : array u8 =
    := out : array u8 = create count
    run_target \|
        | C => \ () => c_read_into path 0 out 1
        | _ => \ () =>
            copy_prefix_from (read_bytes path) out 0
            out

:= u32_le_at (b : array u8) (at : i32) : u32 =
    := byte (k : i32) : u32 = conv (index b (at + k))
    byte 0 ||| (byte 1 <<< 8) ||| (byte 2 <<< 16) ||| (byte 3 <<< 24)

:~ ^ f32s_from (b : array u8) (out : array f32) (offset : i32) (i : i32) : () =
    ? i < length out ?>
        set out i (convert::f32_of_bits (u32_le_at b (offset + 4 * i)))
        f32s_from b out offset (i + 1)

:= read_f32s (path : string) (at : i32) (count : i32) : array f32 =
    := out : array f32 = create count
    run_target \|
        | C => \ () => c_read_into path at out 4
        | _ => \ () =>
            f32s_from (read_bytes path) out at 0
            out

:= c_map_includes () : () =
    global "#include <stdint.h>"
    global (
        "#ifdef _WIN32\n#include <windows.h>\n#else\n#include <sys/mman.h>\n"
        ^^ "#include <fcntl.h>\n#include <unistd.h>\n#endif"
    )

:= c_open_win () : () =
    global (
        "#ifdef _WIN32\nstatic HANDLE spiral_open_win(const char * p) { "
        ^^ "return CreateFileA(p, GENERIC_READ, FILE_SHARE_READ, NULL, OPEN_EXISTING, 0, NULL); }\n#endif"
    )

:= c_sized_win () : () =
    global (
        "#ifdef _WIN32\nstatic HANDLE spiral_sized_win(const char * p, long * n) { HANDLE f = spiral_open_win(p); "
        ^^ "LARGE_INTEGER s; *n = GetFileSizeEx(f, &s) ? (long)s.QuadPart : -1; return f; }\n#endif"
    )

:= c_mapping_win () : () =
    global (
        "#ifdef _WIN32\nstatic HANDLE spiral_mapping_win(HANDLE f, long n) { HANDLE m = n < 0 ? NULL : "
        ^^ "CreateFileMappingA(f, NULL, PAGE_WRITECOPY, 0, 0, NULL); CloseHandle(f); return m; }\n#endif"
    )

:= c_view_win () : () =
    global (
        "#ifdef _WIN32\nstatic char * spiral_view_win(HANDLE m) { "
        ^^ "char * r = m ? (char *)MapViewOfFile(m, FILE_MAP_COPY, 0, 0, 0) : NULL; "
        ^^ "if (m) CloseHandle(m); return r; }\n#endif"
    )

:= c_map_private_win () : () =
    global (
        "#ifdef _WIN32\nstatic char * spiral_map_private(const char * p, long * n) { "
        ^^ "HANDLE f = spiral_sized_win(p, n); return spiral_view_win(spiral_mapping_win(f, *n)); }\n#endif"
    )

:= c_open_sized () : () =
    global (
        "#ifndef _WIN32\nstatic int spiral_open_sized(const char * p, long * n) { int fd = open(p, O_RDONLY); "
        ^^ "if (fd >= 0) *n = (long)lseek(fd, 0, SEEK_END); return fd; }\n#endif"
    )

:= c_mmap () : () =
    global (
        "#ifndef _WIN32\nstatic char * spiral_mmap(int fd, long n) { void * r = mmap(NULL, n, PROT_READ | PROT_WRITE, "
        ^^ "MAP_PRIVATE, fd, 0); close(fd); return r == MAP_FAILED ? NULL : (char *)r; }\n#endif"
    )

:= c_map_private_posix () : () =
    global (
        "#ifndef _WIN32\nstatic char * spiral_map_private(const char * p, long * n) { "
        ^^ "int fd = spiral_open_sized(p, n); return fd < 0 ? NULL : spiral_mmap(fd, *n); }\n#endif"
    )

:= c_map_private_global () : () =
    c_map_includes ()
    c_open_win ()
    c_sized_win ()
    c_mapping_win ()
    c_view_win ()
    c_map_private_win ()
    c_open_sized ()
    c_mmap ()
    c_map_private_posix ()

:= c_pin_header () : () =
    global (
        "static void * spiral_pin_header(char * h, long count) { "
        ^^ "((int32_t *)h)[0] = 1 << 30; ((uint32_t *)h)[1] = (uint32_t)count; return h; }"
    )

:= c_view_f32s () : () =
    c_pin_header ()
    global (
        "static void * spiral_view_f32s(char * m, long n, long at, long count) { "
        ^^ "return !m || at < 8 || (at - 8) % 4 || at + 4 * count > n ? NULL : spiral_pin_header(m + at - 8, count); }"
    )

:= c_map_f32s_global () : () =
    global (
        "__attribute__((noinline)) static void * spiral_map_f32s(const char * p, long at, long count) { "
        ^^ "long n = 0; char * m = spiral_map_private(p, &n); return spiral_view_f32s(m, n, at, count); }"
    )

:= c_map_global () : () =
    c_map_private_global ()
    c_view_f32s ()
    c_map_f32s_global ()

:= c_map_header (path : string) (at : i32) (count : i32) : u64 =
    c_map_global ()
    := h : u64 = $'(uint64_t)(uintptr_t)spiral_map_f32s(!path->ptr, !at, !count)' : u64
    ? h = 0u64 ?> failwith "file_system.map_f32s / cannot map that range"
    h

:= c_map_f32s (path : string) (at : i32) (count : i32) : array f32 =
    := shape : array f32 = create 0
    := h = c_map_header path at count
    $'(__typeof__(!shape))(uintptr_t)!h' : array f32

:= map_f32s (path : string) (offset : i32) (count : i32) : array f32 =
    run_target \|
        | C => \ () => c_map_f32s path offset count
        | _ => \ () => read_f32s path offset count

:= c_write_bytes_global () : () =
    c_stdio ()
    global (
        "static void spiral_write_bytes(const char * p, const void * b, long n) { "
        ^^ "FILE * f = fopen(p, \"wb\"); fwrite(b, 1, (size_t)n, f); fclose(f); }"
    )

:= c_write_bytes (path : string) (b : array u8) : () =
    c_write_bytes_global ()
    $'spiral_write_bytes(!path->ptr, !b->ptr, !b->len)' : ()

:= rust_write_bytes (path : string) (b : array u8) : () =
    := v = b |> am'::base |> am'::to_vec
    $'std::fs::write(&*!path, &*!v).unwrap()' : ()

:= zig_write_bytes (path : string) (b : array u8) : () =
    zig_io ()
    $'std.Io.Dir.cwd().writeFile(spiral_io_threaded.io(), .{ .sub_path = !path, .data = !b }) catch \
        \@panic("file_system.write_bytes failed")' : ()

:= fallback_write_bytes (path : string) (b : array u8) : () =
    run_target \|
        | Zig => \ () => zig_write_bytes path b
        | _ => \ () => failwith "file_system.write_bytes / no arm for this backend"

:= write_bytes (path : string) (b : array u8) : () =
    run_target \|
        | C => \ () => c_write_bytes path b
        | Rust => \ () => rust_write_bytes path b
        | _ => \ () => fallback_write_bytes path b

:= f32_bytes () : array u8 = ;[9u8; 9u8; 9u8; 9u8; 1u8; 0u8; 128u8; 63u8; 0u8; 0u8; 0u8; 192u8]

:= map_bytes () : array u8 = ;[9u8; 9u8; 9u8; 9u8; 9u8; 9u8; 9u8; 9u8; 1u8; 0u8; 128u8; 63u8; 0u8; 0u8; 0u8; 192u8]

#### map_f32s_views_floats_after_an_8_byte_header (test)

In [10]:
///- --test --symbol
///> c
///> rust
///> zig

:= ~path = get_temp_path () ++# "/spiral_map_f32s.bin"
write_bytes path (map_bytes ())
:= x = map_f32s path 8 2
(length x, index x 0, index x 1) |> _assert_eq (2i32, 1.0000001f32, -2.0f32)

.c (c)
__assert_eq { expected = (2, 1.0000001, -2.0) }
.rs (rust)
__assert_eq { expected = (2, 1.0000001, -2.0) }
.zig (zig)
__assert_eq { expected = (2, 1.0000001, -2.0) }

0

#### read_f32s_reads_little_endian_floats_at_an_offset (test)

In [11]:
///- --test --symbol
///> c
///> rust
///> zig

:= ~path = get_temp_path () ++# "/spiral_read_f32s.bin"
write_bytes path (f32_bytes ())
:= x = read_f32s path 4 2
(length x, index x 0, index x 1) |> _assert_eq (2i32, 1.0000001f32, -2.0f32)

.c (c)
__assert_eq { expected = (2, 1.0000001, -2.0) }
.rs (rust)
__assert_eq { expected = (2, 1.0000001, -2.0) }
.zig (zig)
__assert_eq { expected = (2, 1.0000001, -2.0) }

0

#### read_prefix_reads_the_first_bytes (test)

In [12]:
///- --test --symbol
///> c
///> rust
///> zig

:= ~path = get_temp_path () ++# "/spiral_read_prefix.bin"
write_bytes path (f32_bytes ())
:= head = read_prefix path 5
(length head, index head 3, index head 4) |> _assert_eq (5i32, 9u8, 1u8)

.c (c)
__assert_eq { expected = (5, 9, 1) }
.rs (rust)
__assert_eq { expected = (5, 9, 1) }
.zig (zig)
__assert_eq { expected = (5, 9, 1) }

0

In [ ]:
:= read_all_text (path : string) : string =
    run_target \|
        | Zig => \ () =>
            global "var spiral_io_threaded: std.Io.Threaded = .init_single_threaded;"
            $'(std.Io.Dir.cwd().readFileAlloc(spiral_io_threaded.io(), !path, spiral_gpa, .unlimited) catch \
                \@panic("file_system.read_all_text failed"))' : string
        | Fsharp => \ () =>
            $'!path |> System.IO.File.ReadAllText'
        | Gleam => \ () =>
            file_system_gleam_files ()
            gleam_read_text path
        | Rust => \ () =>
            path
            |> read_all_bytes
            |> sm'::string_from_utf8
            |> resultm::unwrap'
            |> sm'::from_std_string
        | Lua => \ () =>
            := file = lua_file_open path "r"
            := text = lua_file_read_all file
            lua_file_close file
            text
        | Pascal => \ () =>
            := handle = $'FileOpen(!path, fmOpenRead)' : i64
            ? handle = -1i64 ?> failwith "file_system.read_all_text / cannot open the file"
            := size = $'FileSeek(!handle, 0, 2)' : i32
            := start = $'FileSeek(!handle, 0, 0)' : i32
            ? start <> 0i32 ?> failwith "file_system.read_all_text / cannot rewind the file"
            := text = $'StringOfChar(Chr(0), !size)' : string
            ? size > 0i32 ?>
                := first = 1i32
                := read = $'FileRead(!handle, (!text)[!first], !size)' : i32
                ? read <> size ?> failwith "file_system.read_all_text / short read"
            $'FileClose(!handle)' : ()
            text
        | _ => \ () => null ()

#### pascal_read_all_text_reads_back_what_was_written (test)

In [13]:
///- --test --symbol
///> pascal

:= path = "spiral-pascal-read-probe.txt"
write_all_text path "ab"
path |> read_all_text |> _assert_eq "ab"
write_all_text path ""
path |> read_all_text |> _assert_eq ""

unsupported on this machine: `dcc32` was not found (nor fpc)

In [ ]:
:= read_text_or_empty (path : string) : string =
    run_target \|
        | Rust => \ () =>
            $'std::rc::Rc::<str>::from(std::fs::read_to_string(&*!path).unwrap_or_default().as_str())' : string
        | Fsharp => \ () =>
            $'(try System.IO.File.ReadAllText(!path : string) with _ -> "")' : string
        | _ => \ () => no_arm "file_system.read_text_or_empty"

:= read_text_or_empty_probe () : string =
    := path = get_temp_path () </> "spiral-read-text-or-empty.txt"
    write_all_text path "kept"
    path

#### read_text_or_empty_reads_a_file_or_gives_empty (test)

In [14]:
///- --test --symbol
///> rust

:= path = read_text_or_empty_probe ()
(read_text_or_empty path, read_text_or_empty (path ++# ".missing")) |> _assert_eq ("kept", "")

__assert_eq { expected = (kept, ) }

0

In [ ]:
:= directory_create_symbolic_link (target : string) (path : string) : () =
    run_target \|
        | Fsharp => \ () =>
            ($'System.IO.Directory.CreateSymbolicLink (!path, !target)' : file_system_info)
            |> ignore
        | Rust => \ () =>
            global """
                #[cfg(windows)]
                fn spiral_fs_symlink_dir(target: &str, path: &str) -> std::io::Result<()> {
                    std::os::windows::fs::symlink_dir(target, path)
                }
                #[cfg(unix)]
                fn spiral_fs_symlink_dir(target: &str, path: &str) -> std::io::Result<()> {
                    std::os::unix::fs::symlink(target, path)
                }
                #[cfg(not(any(windows, unix)))]
                fn spiral_fs_symlink_dir(_target: &str, _path: &str) -> std::io::Result<()> {
                    Err(std::io::ErrorKind::Unsupported.into())
                }
                """
            := linked : resultm::result' () stream::io_error = $'spiral_fs_symlink_dir(&*!target, &*!path)'
            ?= linked |> resultm::map_error' sm'::format |> resultm::unbox ?|
            | Ok () => ()
            | Error why => failwith ("directory_create_symbolic_link / " ++# path ++# ": " ++# why)
        | _ => \ () => ()

:= file_create_symbolic_link (target : string) (path : string) : () =
    run_target \|
        | Fsharp => \ () =>
            ($'System.IO.File.CreateSymbolicLink (!path, !target)' : file_system_info)
            |> ignore
        | Rust => \ () =>
            global """
                #[cfg(windows)]
                fn spiral_fs_symlink_file(target: &str, path: &str) -> std::io::Result<()> {
                    std::os::windows::fs::symlink_file(target, path)
                }
                #[cfg(unix)]
                fn spiral_fs_symlink_file(target: &str, path: &str) -> std::io::Result<()> {
                    std::os::unix::fs::symlink(target, path)
                }
                #[cfg(not(any(windows, unix)))]
                fn spiral_fs_symlink_file(_target: &str, _path: &str) -> std::io::Result<()> {
                    Err(std::io::ErrorKind::Unsupported.into())
                }
                """
            := linked : resultm::result' () stream::io_error = $'spiral_fs_symlink_file(&*!target, &*!path)'
            ?= linked |> resultm::map_error' sm'::format |> resultm::unbox ?|
            | Ok () => ()
            | Error why => failwith ("file_create_symbolic_link / " ++# path ++# ": " ++# why)
        | _ => \ () => ()

:= mklink_status (native_path : string) (native_target : string) : i32 =
    global """
        fn spiral_fs_mklink_junction(path: &str, target: &str) -> i32 {
            let output = std::process::Command::new("cmd").args(["/d", "/c", "mklink", "/J", path, target]).output();
            output.map(|o| o.status.code().unwrap_or(-2)).unwrap_or(-1)
        }
        """
    $'spiral_fs_mklink_junction(&*!native_path, &*!native_target)' : i32

:= directory_create_junction (target : string) (path : string) : result () string =
    ? path |> directory_exists ?> Error ("directory_create_junction / path exists: " ++# path)
    |:
        run_target \|
            | Rust => \ () =>
                ? platform::is_windows () ?>
                    := native_path = path |> sm'::replace "/" "\\"
                    := native_target = target |> sm'::replace "/" "\\"
                    := exit_code = mklink_status native_path native_target
                    := failure = "directory_create_junction / mklink /J exit " ++# sm'::format exit_code
                    ? exit_code = 0i32 ?> Ok () |: Error failure
                |:
                    path |> directory_create_symbolic_link target
                    Ok ()
            | _ => \ () =>
                path |> directory_create_symbolic_link target
                Ok ()

:= file_is_read_only (path : string) : bool =
    run_target \|
        | Rust => \ () => $'std::fs::metadata(&*!path).map(|m| m.permissions().readonly()).unwrap_or(false)' : bool
        | _ => \ () => !-

:! fs_permissions = $'std::fs::Permissions'

:= rust_permissions (path : string) : optionm'::option' fs_permissions =
    $'std::fs::metadata(&*!path).ok().map(|m| m.permissions())' : optionm'::option' fs_permissions

:= rust_permissions_set (path : string) (read_only : bool) (mode : fs_permissions) : bool =
    := mode = mode
    mode |> rust::to_mut
    $'!mode.set_readonly(!read_only)' : ()
    $'std::fs::set_permissions(&*!path, !mode).is_ok()' : bool

:= file_set_read_only (read_only : bool) (path : string) : bool =
    run_target \|
        | Rust => \ () =>
            ?= rust_permissions path |> optionm'::unbox ?|
            | Some permissions => rust_permissions_set path read_only permissions
            | None => !-
        | _ => \ () => !-

:| entry_kind =
    | File
    | Directory

:~ find_parent file_type name root_dir =
    := is_file = file_type = File
    := exists_fn = ? is_file ?> file_exists |: directory_exists
    :~ ^ 루프 dir =
        ? dir </> name |> exists_fn
        ?> dir |> Ok
        |:
            ?= dir |> directory_parent ?|
            | Some parent => parent |> 루프
            | None =>
                "file_system.find_parent / No parent for " ++# (? is_file ?> "file" |: "dir")
                ++# " '" ++# name ++# "' at '" ++# root_dir ++# "' (until '" ++# dir ++# "')"
                |> Error
    root_dir |> 루프

#### find_parent (test)

In [15]:
///- --test --symbol
///> fsharp

;[ Directory, "workspace"; File, "README.md" ]
|> arraym::map \ file_type, file =>
    get_source_directory ()
    |> find_parent file_type file
    |> resultm::get
    |> directory_info
    |> directory_info_name
|> am'::distinct
|> _assert_in [ ;[ "spiral" ]; ;[ "polyglot" ] ]

17:42:42 1d _assert_in / { x = [spiral]; x' = [spiral] }

In [16]:
///- --test --symbol
///> rust

;[ Directory, "workspace"; File, "README.md" ]
|> arraym::map \ file_type, file =>
    \ () =>
        ~~
            get_current_directory ()
            |> find_parent file_type file
            |> resultm::get
            |> sm'::to_std_string
            |> new_path_buf
            |> path_buf_file_name
            |> optionm'::try'
            |> sm'::from_os_str_ref
            |> Some
            |> optionm'::box
    |> \ x => x () |> optionm'::unbox
    |> optionm'::default_value ""
|> am'::distinct
|> \ result =>
    result |> am'::length |> _assert_eq 1i32
    index result 0i32 |> _assert_eq "spiral"

__assert_eq { expected = 1 }
__assert_eq { expected = spiral }

0

In [ ]:
:= file_delete (path : string) : () =
    run_target \|
        | Zig => \ () =>
            global "var spiral_io_threaded: std.Io.Threaded = .init_single_threaded;"
            $'std.Io.Dir.cwd().deleteFile(spiral_io_threaded.io(), !path) catch {}' : ()
        | Fsharp => \ () =>
            path |> $'System.IO.File.Delete'
        | Rust => \ () =>
            := result : resultm::result' () stream::io_error =
                ? path |> file_exists
                ?> !\\(path, $'"std::fs::remove_file(&*$0)"')
                |: Ok () |> resultm::box
            ?= result |> resultm::map_error' sm'::format |> resultm::unbox ?|
            | Ok () => ()
            | Error error' => console::write_error_line (sm'::(++#) "file_system.file_delete: " error')
        | Gleam => \ () =>
            file_system_gleam_paths ()
            $'spiral_fs_delete(!path)' : ()
        | Lua => \ () => lua_file_remove path
        | _ => \ () => ()

:= written_then_deleted (path : string) : bool * bool =
    write_all_text path "x"
    := written = file_exists path
    file_delete path
    written, file_exists path

#### file_exists_on_gleam_tells_files_from_missing_paths (test)

In [17]:
///- --test --symbol
///> gleam

:= ~path = "spiral-file-exists-probe.txt"
(written_then_deleted path, file_exists "spiral-no-such-file.txt") |> _assert_eq ((!+, !-), !-)

__assert_eq { expected = ((true, false), false) }

In [ ]:
:~ read_link (path : string) : resultm::result' path_buf stream::io_error =
    :~ run 루프 n error path' =
        := name = path' |> get_file_name
        := parent = path' |> directory_get_parent |> optionm'::unbox
        := error'' = #?error
        ?= parent ?|
        | _ ?& n >= 11 =>
            ($'$"file_system.read_link / "' : string)
            ++# $'$"path: {!path} / n: {!n} / path\': {!path'} / name: {!name}"'
            |> stream::new_io_error
            |> resultm::err
        | Some parent ?& path' <>. "" =>
            ?= 루프 (n + 1) parent |> resultm::map_error' sm'::format |> resultm::unbox ?|
            | Ok parent' =>
                (parent' |> path_buf_display |> convert::convert) </> name
                |> sm'::to_std_string
                |> new_path_buf
                |> resultm::ok''
            | Error error' =>
                ($'$"file_system.read_link / "' : string)
                ++# $'$"error\': {!error'} / error: {!error''} / name: {!name}"'
                |> stream::new_io_error
                |> resultm::err
        | _ =>
            ($'$"file_system.read_link / run / The file or directory is not a reparse point. / "' : string)
            ++# $'$"path: {!path} / error: {!error''} / path\': {!path'} / name: {!name}"'
            |> stream::new_io_error
            |> resultm::err

    run_target \|
        | Rust => \ () =>
            ? path |> directory_exists
            ?> !\\(path, $'"std::fs::read_link(&*$0)"')
            |:
                :~ ^ 루프 \* t {number}.
                    (n : t)
                    (path' : string)
                    : resultm::result' path_buf stream::io_error
                    =
                    run_target \|
                        | Rust => \ () =>
                            := result : _ _ stream::io_error = !\\(path', $'"std::fs::read_link(&*$0)"')
                            := result = result |> resultm::map_error' sm'::format |> resultm::unbox
                            ?= result ?|
                            | Ok x => x |> resultm::ok''
                            | Error error => path' |> run 루프 n error
                        | _ => \ () => null ()
                path |> 루프 0u8
        | Fsharp _
        | TypeScript _ => \ () =>
            :~ ^ 루프 \* t {number}.
                (n : t)
                (path' : string)
                : resultm::result' path_buf stream::io_error
                =
                := result =
                    path'
                    |> directory_info
                    |> directory_info_attributes
                    |> file_attributes_has_flag (file_attributes_reparse_point ())
                ? result ?>
                    path'
                    |> file_info
                    |> file_info_link_target
                    |> convert::unbox
                    |> resultm::ok''
                |:
                    := error =
                        ($'$"file_system.read_link / Fsharp / "' : string)
                        ++# $'$"The file or directory is not a reparse point. / "'
                        ++# $'$"path: {!path} / result: {!result} / path\': {!path'} / n: {!n}"'
                        |> stream::new_io_error
                    path' |> run 루프 n error
            path |> 루프 0u8
        | _ => \ () => $'Unchecked.defaultof<_>'

:~ normalize_path (path : string) : string =
    ? path = ""
    ?> ""
    |:
        := result =
            ?= path |> read_link |> resultm::ok' |> optionm'::unbox ?|
            | Some path_buf =>
                path_buf
                |> path_buf_display
                |> convert::convert
            | None => path
        := result =
            ? result = ""
            ?> path
            |: result
        := result' =
            ? result |> sm'::starts_with "\\\\?\\"
            ?> result |> sm'::substring 4i32 ((sm'::length result : i32) - 4i32)
            |: result
        ? ##result' < 2
        ?> path
        |:
            sm'::(++#)
                (result' |> sm'::slice 0i32 0i32 |> sm'::to_lower)
                (result' |> sm'::slice 1i32 ((sm'::length result' : i32) - 1i32))
            |> sm'::replace "\\" "/"

#### normalize_path_drops_the_verbatim_prefix (test)

In [18]:
///- --test --symbol
///> rust

:= ~path = "\\\\?\\C:\\x"
path |> normalize_path |> _assert_eq "c:/x"

__assert_eq { expected = c:/x }

0

In [ ]:
:= deps_workspace_root (root : string) : string =
    ?= root |> get_file_name ?|
    | "deps" =>
        ?= root |> directory_parent ?|
        | Some parent =>
            ?= parent |> find_parent Directory ("spiral" </> "workspace") ?|
            | Ok outer => outer |> normalize_path
            | Error _ => root
        | None => root
    | _ => root

#### a_deps_root_without_an_outer_checkout_keeps_its_own_spiral (test)

In [19]:
///- --test --symbol
///> rust

:= deps = get_temp_path () </> "deps_workspace_root_test" </> "deps"
:= plain = get_temp_path () </> "deps_workspace_root_test"
(deps_workspace_root deps = deps, deps_workspace_root plain = plain) |> _assert_eq (!+, !+)

__assert_eq { expected = (true, true) }

0

In [ ]:
:= get_workspace_root () =
    := repo_root = env::get_environment_variable "SPIRAL_REPO_ROOT"
    ? repo_root <> ""
    ?> repo_root |> normalize_path
    |:
        := dirs =
            [ get_current_directory; get_source_directory; \ () => "/workspaces" ]
        :~ run dir =
            dir
            |> find_parent Directory ("spiral" </> "workspace")
            |> \|
                | Ok dir => dir |> normalize_path |> Some
                | Error error =>
                    trace Warning
                        \ () => "file_system.get_workspace_root"
                        \ () => { dir error }
                    None
        := root =
            (None, dirs)
            ||> listm::fold \ acc dir =>
                ?= acc ?|
                | Some dir => Some dir
                | None => dir () |> run
            |> optionm::value
        deps_workspace_root root </> "spiral"

:= get_workspace_root_external () =
    := workspace_root = get_workspace_root ()
    := current_dir = get_current_directory () |> sm'::to_lower
    := workspace_root = workspace_root |> sm'::to_lower
    ? current_dir |> sm'::starts_with workspace_root
    ?> Error workspace_root
    |: Ok workspace_root

#### get_workspace_root_external (test)

In [20]:
///- --test --symbol
///> rust -d regex

?= get_workspace_root_external () ?|
| Ok root => root
| Error root => root
|> get_file_name
|> _assert_eq "spiral"

__assert_eq { expected = spiral }

In [ ]:
:~ get_full_path (path : string) : string =
    run_target_args (\ () => path) \|
        | Fsharp => \ path =>
            path |> $'System.IO.Path.GetFullPath'
        | Rust => \ path =>
            := current_dir = get_current_directory ()
            current_dir </> path
            |> normalize_path
            |> sm'::split "/"
            |> \ x =>
                ((array x : array _), (0i32, (;[] : array _)))
                ||> arraym::foldBack \ x level, acc =>
                    ?= x, level ?|
                    | "..", _ => level + 1, acc
                    | ".", _ => level, acc
                    | _, 0 ?& x |> sm'::ends_with ":" =>
                        := drive = current_dir |> sm'::slice 0i32 0i32
                        := letter = x |> sm'::slice 0i32 0i32
                        0, ;[ sm'::(++#) (? sm'::to_lower letter = sm'::to_lower drive ?> drive |: sm'::to_upper letter) ":" ] ++ acc
                    | _, 0 => 0, ;[ x ] ++ acc
                    | _ => level - 1, acc
            |> snd
            |> seq::of_array'
            |> sm'::concat (directory_separator_char () |> sm'::obj_to_string)
        | Gleam => \ path =>
            file_system_gleam_paths ()
            gleam_lexical_full_path path
        | _ => \ _ => null ()

#### get_full_path (test)

In [21]:
///- --test --symbol
///> fsharp

"."
|> get_full_path
|> directory_info
|> directory_info_name
|> _assert_in [ "spiral"; "fsharp" ]

17:42:50 1d _assert_in / { x = spiral; x' = spiral }

In [22]:
///- --test --symbol
///> rust -d regex

"dir/.././._file"
|> get_full_path
|> _assert_eq (get_current_directory () </> "._file")

__assert_eq { expected = /home/runner/work/spiral/spiral/lib/spiral/._file }

#### get_full_path_keeps_the_drive_of_an_absolute_path (test)

In [23]:
///- --test --symbol
///> rust -d regex

? platform::is_windows () ?>
    := drive = ? get_current_directory () |> sm'::starts_with "Z" ?> "Y" |: "Z"
    sm'::(++#) drive ":/spiral_drive_probe/x"
    |> get_full_path
    |> _assert_eq (sm'::(++#) drive ":\\spiral_drive_probe\\x")

#### get_full_path_3 (test)

In [24]:
///- --test --symbol
///> rust -d regex

:= directory_name path =
    path
    |> sm'::to_std_string
    |> new_path_buf
    |> path_buf_file_name
    |> optionm'::unwrap
    |> sm'::from_os_str_ref
"."
|> get_full_path
|> directory_name
|> _assert_eq (get_current_directory () |> directory_name)

__assert_eq { expected = spiral }

#### get_full_path_4 (test)

In [25]:
///- --test --symbol
///> rust -d regex

"dir/.././._file"
|> get_full_path
|> normalize_path
|> _assert_eq (get_current_directory () </> "._file" |> normalize_path)

__assert_eq { expected = /home/runner/work/spiral/spiral/lib/spiral/._file }

In [ ]:
:~ standardize_path path =
    path |> get_full_path |> normalize_path

:~ absolute_path path =
    := current_dir = get_current_directory ()
    current_dir </> path |> standardize_path

:= new_file_uri (path : string) : string =
    := path = path |> sm'::trim_start [ '/' ]
    "file:///" ++# path

#### new_file_uri (test)

In [26]:
///- --test --symbol
///> rust -d regex

@"\\?\C:\test"
|> normalize_path
|> new_file_uri
|> _assert_eq "file:///c:/test"

__assert_eq { expected = file:///c:/test }

In [27]:
///- --test --symbol
///> rust -d regex

@"\\?\C:\test"
|> normalize_path
|> new_file_uri
|> _assert_eq "file:///c:/test"

__assert_eq { expected = file:///c:/test }

## fsharp

In [ ]:
:~ ^ trace_file text : () =
    run_target \|
    | Fsharp => \ () =>
        resultm::from_try
            \ () =>
                := assembly_name = env::get_entry_assembly_name ()
                := guid = date_time::now () |> date_time::new_guid_from_date_time
                := file_name = assembly_name ++# "_" ++# sm'::obj_to_string guid ++# ".txt"

                := workspace_root = get_workspace_root ()
                := trace_dir = workspace_root </> "target/trace"
                trace_dir |> create_directory |> ignore
                := path = trace_dir </> file_name
                text |> write_all_text_async path |> async::run_synchronously
            \ ex =>
                := ex = ex ()
                := text = $'$"file_system.trace_file / ex: %A{!ex}"'
                text |> console::write_line
                text |> trace_file
                ex
        |> resultm::get
    | _ => \ () => ()

#### trace_file (test)

In [28]:
///- --test --symbol
///> fsharp

:= get_count dir : i64 =
    := files = dir |> directory_get_files
    array files |> am'::length |> conv

:= trace_dir = get_workspace_root () </> "target/trace"
trace_dir |> create_directory |> ignore

:= count = get_count trace_dir

trace_file "test"

get_count trace_dir
|> _assert_eq (count + 1)

__assert_eq { expected = 1 }

## file_system

In [ ]:
:~ delete_comparable_path (path : string) : string =
    path |> get_full_path |> sm'::replace "\\" "/" |> sm'::trim_end ['/'] |> sm'::to_lower

:= delete_target_contains (target : string) (root : string) : bool =
    root <> "" && (target = root || sm'::starts_with (target ++# "/") root)

:~ delete_shape_protected (target : string) : bool =
    := parts = sm'::split "/" target |> array
    := count = am'::length parts
    := part (at : i32) : string = ? at < count ?> am'::index at parts |: ""
    := rooted_home = part 0i32 = "" && (count = 2i32 || (count = 3i32 && part 1i32 = "home"))
    := app_data = part 3i32 = "appdata" && (count <= 5i32 || (count = 6i32 && part 4i32 = "local" && part 5i32 = "temp"))
    := users = part 1i32 = "users" && (count <= 3i32 || app_data)
    count <= 1i32 || rooted_home || users || (count = 2i32 && part 1i32 = "windows")

:~ ^ delete_variable_protected (target : string) (names : list string) : bool =
    ?= names ?|
    | [] => !-
    | name :: rest =>
        := value = env::get_environment_variable name
        ? value <> "" && delete_target_contains target (delete_comparable_path value) ?> !+
        |: delete_variable_protected target rest

:= delete_target_is_protected (path : string) : bool =
    := variables = ["USERPROFILE"; "HOME"; "TEMP"; "TMP"; "TMPDIR"; "LOCALAPPDATA"; "APPDATA"; "SystemRoot"; "ProgramData"]
    := target = delete_comparable_path path
    path = "" || delete_shape_protected target || delete_variable_protected target variables

#### delete_target_is_protected_refuses_kits_cases_on_windows (test)

In [29]:
///- --test --symbol
///> rust

? platform::is_windows ()
?>
    (
        delete_target_is_protected "C:\\",
        delete_target_is_protected "C:\\Users",
        delete_target_is_protected "C:\\Users\\me",
        delete_target_is_protected "c:/users/someone",
        delete_target_is_protected "C:\\Users\\me\\AppData",
        delete_target_is_protected "C:\\Users\\me\\AppData\\Local\\Temp",
        delete_target_is_protected "C:\\Windows",
        delete_target_is_protected "",
        delete_target_is_protected "C:\\home\\git\\kit\\.cache\\x",
        delete_target_is_protected (get_temp_path () </> "kit-run-42")
    )
    |> _assert_eq (!+, !+, !+, !+, !+, !+, !+, !+, !-, !-)

0

#### delete_target_is_protected_covers_roots_profiles_and_temp (test)

In [30]:
///- --test --symbol
///> rust

:= home =
    ? platform::is_windows ()
    ?> env::get_environment_variable "USERPROFILE"
    |: env::get_environment_variable "HOME"
:= scratch = get_temp_path () </> "spiral-delete-guard-probe"
(
    delete_target_is_protected home,
    delete_target_is_protected (home </> ".."),
    delete_target_is_protected (get_temp_path ()),
    delete_target_is_protected (home </> "AppData" </> ".." </> ".."),
    delete_target_is_protected scratch
)
|> _assert_eq (!+, !+, !+, !+, !-)

__assert_eq { expected = (true, true, true, true, false) }

0

In [ ]:
:~ directory_delete recursive (path : string) : () =
    run_target_args' (path, recursive) \|
        | Fsharp => \ path, recursive =>
            ? recursive && delete_target_is_protected path
            ?> "file_system.directory_delete: refusing a drive root, profile, temp root or system folder: " ++# path |> failwith
            $'System.IO.Directory.Delete (!path, !recursive)'
        | Rust => \ path, recursive =>
            ? recursive && delete_target_is_protected path
            ?> "file_system.directory_delete: refusing a drive root, profile, temp root or system folder: " ++# path |> failwith
            ? path |> directory_exists ?>
                ? recursive
                ?> !\\(path, $'"std::fs::remove_dir_all(&*$0).unwrap()"')
                |: !\\(path, $'"std::fs::remove_dir(&*$0).unwrap()"')
        | _ => \ _ => ()

#### directory_delete_removes_a_scratch_tree (test)

In [31]:
///- --test --symbol
///> rust

:= scratch = get_temp_path () </> "spiral-delete-guard-scratch"
create_dir_all (scratch </> "inner") |> ignore
directory_delete !+ scratch
scratch |> directory_exists
|> _assert_eq !-

__assert_eq { expected = false }

0

#### directory_delete_refuses_a_protected_surrogate_root (test)

In [32]:
///- --test --symbol
///> rust

:= surrogate = get_temp_path () </> "spiral-delete-guard-surrogate"
create_dir_all (surrogate </> "inner") |> ignore
:= before = env::environment_snapshot ()
env::environment_restored (before ++# sm'::char_text 0u8 ++# "LOCALAPPDATA=" ++# surrogate) "SPIRAL_KEPT_"
:= refused =
    catch_unwind_rust
        (\ () =>
            directory_delete !+ surrogate
            !-
        )
        \ _ => !+
env::environment_restored before "SPIRAL_KEPT_"
:= kept = surrogate |> directory_exists
directory_delete !+ surrogate
(refused, kept)
|> _assert_eq (!+, !+)

__assert_eq { expected = (true, true) }

0

In [ ]:
:~ create_dir dir =
    run_target_args' dir \|
        | Rust => \ dir =>
            ?= dir |> create_dir_all |> resultm::map_error' sm'::format' |> resultm::unbox ?|
            | Ok () =>
                trace Verbose
                    \ () => "file_system.create_dir"
                    \ () => { dir }
            | Error error =>
                trace Critical
                    \ () => "file_system.create_dir"
                    \ () => { dir error }
            := disposable : _ () = new_disposable \ () =>
                dir
                |> directory_delete !+
            disposable
        | _ => \ _ => null ()

:= create_temp_directory () =
    := dir = create_temp_path ()
    dir, dir |> create_dir

#### create_temp_directory (test)

In [33]:
///- --test --symbol
///> rust -d chrono

:= path, disposable = create_temp_directory ()
~~
    path
    |> directory_exists
    |> _assert_eq !+
    disposable |> use |> ignore
    path
    |> directory_exists
    |> _assert_eq !+
path
|> directory_exists
|> _assert_eq !-

17:43:24 1v file_system.create_dir / { dir = /tmp/!create_temp_path_/spiral_388838dce00db8c1b04273dd3ea08380245791533f0ba4ec6716a35bd69bcdba/20261107-1743-2440-3517-00000045663f }
__assert_eq { expected = true }
__assert_eq { expected = true }
__assert_eq { expected = false }

In [ ]:
:= create_temp_directory' (hash : string) =
    := dir = hash |> guid::hash_guid |> create_temp_path'
    dir, dir |> create_dir

:~ link_directory target_path path =
    ? target_path |> directory_exists |> not
    ?> target_path |> create_dir |> ignore

    := lib_dir_path = path |> directory_get_parent |> optionm'::default_value' ""
    ? lib_dir_path |> directory_exists |> not
    ?> lib_dir_path |> create_dir |> ignore

    ? (path |> directory_exists)
        && (path |> read_link |> resultm::is_err)
    ?> path |> directory_delete !+

    ? path |> directory_exists |> not
    ?> path |> directory_create_symbolic_link target_path

:~ link_file target_path path =
    ? (path |> file_exists)
        && (path |> read_link |> resultm::is_err)
    ?> path |> file_delete

    ? path |> file_exists |> not
    ?> path |> file_create_symbolic_link target_path

#### link_file (test)

In [34]:
///- --test --symbol
///> rust -d sha2 regex

:= file_name = "LICENSE"
:= text = file_name

:= test_hash =
    #?*(file_name, text)
    |> crypto::hash_text

:= workspace_root = get_workspace_root ()
:= test_dir = workspace_root </> "target/test/file_system" </> test_hash

:= disposable = test_dir |> create_dir
disposable |> use |> ignore

:= dir_path = test_dir </> "dir1"

? dir_path |> directory_exists
?> dir_path |> directory_delete !+

dir_path |> create_dir |> ignore

:= path = dir_path </> file_name
text |> write_all_text path

:= dir_link_path = test_dir </> "link1"

dir_link_path |> link_directory dir_path

:= link_path = dir_link_path </> file_name

link_path
|> read_all_text
|> _assert_eq text

dir_link_path
|> read_link
|> resultm::unwrap'
|> path_buf_display
|> convert::convert
|> _assert sm'::ends_with "dir1"

link_path
|> read_link
|> resultm::unwrap'
|> path_buf_display
|> convert::convert
|> _assert sm'::ends_with "LICENSE"

:= link_name = "LICENSE_"

:= link_path = dir_path </> link_name

link_path |> link_file path

:= link_path' = dir_link_path </> link_name

link_path'
|> read_all_text
|> _assert_eq text

link_path
|> read_link
|> resultm::unwrap'
|> path_buf_display
|> convert::convert
|> _assert sm'::ends_with "LICENSE"

link_path'
|> read_link
|> resultm::unwrap'
|> path_buf_display
|> convert::convert
|> _assert sm'::ends_with "LICENSE"

17:43:27 1v file_system.create_dir / { dir = /home/runner/work/spiral/spiral/target/test/file_system/d2a575f48045e269d7c0d7a17d801aad61db34f865675c516636a3bb2c28cd38 }
17:43:27 2v file_system.create_dir / { dir = /home/runner/work/spiral/spiral/target/test/file_system/d2a575f48045e269d7c0d7a17d801aad61db34f865675c516636a3bb2c28cd38/dir1 }
__assert_eq { expected = LICENSE }
__assert { expected = /home/runner/work/spiral/spiral/target/test/file_system/d2a575f48045e269d7c0d7a17d801aad61db34f865675c516636a3bb2c28cd38/dir1 }
__assert { expected = /home/runner/work/spiral/spiral/target/test/file_system/d2a575f48045e269d7c0d7a17d801aad61db34f865675c516636a3bb2c28cd38/dir1/LICENSE }
__assert_eq { expected = LICENSE }
__assert { expected = /home/runner/work/spiral/spiral/target/test/file_system/d2a575f48045e269d7c0d7a17d801aad61db34f865675c516636a3bb2c28cd38/dir1/LICENSE }
__assert { expected = /home/runner/work/spiral/spiral/target/test/file_system/d2a575f48045e269d7c0d7a17d801aad61db34f865675

#### directory_create_junction_links_reads_and_refuses_a_taken_path (test)

In [35]:
///- --test --symbol
///> rust -d sha2 regex

:= test_dir = get_workspace_root () </> "target/test/file_system/junction"
? test_dir |> directory_exists ?> test_dir |> directory_delete !+
:= target = test_dir </> "target"
target |> create_dir |> ignore
"through the junction" |> write_all_text (target </> "hello.txt")
:= link = test_dir </> "link"
:= first = directory_create_junction target link
:= second = directory_create_junction target link
:= read = (link </> "hello.txt") |> read_all_text
(first = Ok (), second <> Ok (), read) |> _assert_eq (!+, !+, "through the junction")

17:43:30 1v file_system.create_dir / { dir = /home/runner/work/spiral/spiral/target/test/file_system/junction/target }
__assert_eq { expected = (true, true, through the junction) }

#### file_read_only_round_trips (test)

In [36]:
///- --test --symbol
///> rust -d sha2 regex

:= test_dir = get_workspace_root () </> "target/test/file_system/read_only"
test_dir |> create_dir |> ignore
:= path = test_dir </> "blob.txt"
? path |> file_exists ?> file_set_read_only !- path |> ignore
"blob" |> write_all_text path
:= before = file_is_read_only path
:= locked = file_set_read_only !+ path
:= after = file_is_read_only path
:= unlocked = file_set_read_only !- path
(before, locked, after, unlocked, file_is_read_only path) |> _assert_eq (!-, !+, !+, !+, !-)

17:43:38 1v file_system.create_dir / { dir = /home/runner/work/spiral/spiral/target/test/file_system/read_only }
__assert_eq { expected = (false, true, true, true, false) }

## rust

In [ ]:
:~ file_exists_content path content : bool =
    run_target \|
        | Rust => \ () =>
            ? path |> file_exists |> not
            ?> !-
            |:
                := existing_content = path |> read_all_text
                content = existing_content
        | _ => \ () => null ()

:~ write_all_text_exists path contents =
    := exists' = contents |> file_exists_content path
    ? not exists' ?>
        := dir = path |> directory_get_parent |> optionm'::default_value' ""
        ? dir |> directory_exists |> not
        ?> dir |> create_dir |> ignore
        contents |> write_all_text path

## main

In [ ]:
///> fsharp

:= main () =
    init_trace_state None
    $'let file_exists_content x = !file_exists_content x' : ()
    $'let write_all_text_async x = !write_all_text_async x' : ()
    $'let create_temp_path () = !create_temp_path ()' : ()
    $'let create_temp_dir () = !create_temp_directory ()' : ()
    $'let create_temp_dir\' x = !create_temp_directory' x' : ()
    $'let get_source_directory () = !get_source_directory ()' : ()
    $'let normalize_path x = !normalize_path x' : ()
    $'let get_full_path x = !get_full_path x' : ()
    $'let standardize_path x = !standardize_path x' : ()
    $'let new_file_uri x = !new_file_uri x' : ()
    $'let get_workspace_root () = !get_workspace_root ()' : ()
    $'let link_directory x = !link_directory x' : ()
    := combine x = (</>) x
    $'let (</>) x = !combine x' : ()

:= remove_dir_all (path : string) : resultm::result' () stream::io_error =
    ? delete_target_is_protected path
    ?> "file_system.remove_dir_all: refusing a drive root, profile, temp root or system folder: " ++# path |> failwith
    !\\(path, $'"std::fs::remove_dir_all(&*$0)"')

:= copy_file (to : string) (path : string) : resultm::result' u64 stream::io_error =
    !\($'"std::fs::copy(&*!path, &*!to)"')

:= canonicalize (path : string) : resultm::result' path_buf stream::io_error =
    !\\(path, $'"std::fs::canonicalize(&*$0)"')

:= set_current_dir (path : string) : resultm::result' () stream::io_error =
    !\\(path, $'"std::env::set_current_dir(&*$0)"')

:= get_current_dir () : resultm::result' path_buf stream::io_error =
    !\($'"std::env::current_dir()"')

:= path_join_native (dir : string) (name : string) : string =
    run_target_args' (dir, name) \|
        | Rust => \ dir, name =>
            := joined : sm'.std_string = !\\((dir, name), $'"std::path::Path::new(&*$0).join(&*$1).display().to_string\
                ()"')
            sm'::from_std_string joined
        | Fsharp => \ dir, name => $'System.IO.Path.Combine (!dir, !name)' : string
        | _ => \ dir, name => dir </> name

#### path_join_native_uses_the_platform_separator (test)

In [37]:
///- --test --symbol
///> rust

:= joined = path_join_native "dir" "file"
(joined = "dir/file" || joined = "dir\\file") |> _assert_eq !+

__assert_eq { expected = true }

0

In [ ]:
:= read_directory_names (path : string) : resultm::result' (am'::vec sm'::std_string) stream::io_error =
    !\\(path, $'"std::fs::read_dir(&*$0).map(|entries| entries.filter_map(|entry| entry.ok().map(|entry| \
        entry.file_name().to_string_lossy().into_owned())).collect::<Vec<std::string::String>>())"')

:= read_dir_names (path : string) : resultm::result' (am'::vec sm'::std_string) stream::io_error = read_directory_names path

#### copy_list_canonicalize_and_remove_a_tree (test)

In [38]:
///- --test --symbol
///> rust -d chrono

:= root, disposable = create_temp_directory ()
~~
    := tree = root </> "tree"
    tree |> create_dir_all |> resultm::unwrap'
    "x" |> write_all_text (tree </> "a.txt")
    := copied = tree </> "a.txt" |> copy_file (tree </> "b.txt") |> resultm::unwrap'
    := names = tree |> read_directory_names |> resultm::unwrap' |> am'::vec_len |> convert::i32
    := canonical_ok = ?= tree |> canonicalize |> resultm::unbox ?| Ok _ => !+ | Error _ => !-
    tree |> remove_dir_all |> resultm::unwrap'
    (copied, names, canonical_ok, tree |> directory_exists) |> _assert_eq (1u64, 2i32, !+, !-)
    disposable |> use |> ignore

17:43:42 1v file_system.create_dir / { dir = /tmp/!create_temp_path_/spiral_4b124722f9dd3a4abf1e8b2d216dcd4d8fdbc123fde628fa3eca5cb93bbb259d/20261107-1743-4243-6534-300000398fcd }
__assert_eq { expected = (1, 2, true, false) }

In [ ]:
:! rust_metadata_found = $'std::io::Result<std::fs::Metadata>'

:! rust_metadata = $'std::fs::Metadata'

:! rust_modified_found = $'std::io::Result<std::time::SystemTime>'

:! rust_system_time = $'std::time::SystemTime'

:! rust_elapsed = $'std::time::Duration'

:! rust_file_opened = $'std::io::Result<std::fs::File>'

:! rust_open_file = $'std::fs::File'

:! rust_io_done = $'std::io::Result<()>'

:= rust_metadata_of (path : string) : rust_metadata_found = $'std::fs::metadata(&*!path)'

:= rust_metadata_ok (r : rust_metadata_found) : bool = $'!r.is_ok()'

:= rust_metadata_value (r : rust_metadata_found) : rust_metadata = $'!r.unwrap()'

:= rust_modified_of (m : rust_metadata) : rust_modified_found = $'!m.modified()'

:= rust_modified_ok (r : rust_modified_found) : bool = $'!r.is_ok()'

:= rust_modified_value (r : rust_modified_found) : rust_system_time = $'!r.unwrap()'

:= rust_since_epoch (t : rust_system_time) : rust_elapsed = $'!t.duration_since(std::time::UNIX_EPOCH).unwrap_or_defau\
    lt()'

:= rust_elapsed_ms (d : rust_elapsed) : u64 = ($'!d.as_secs()' : u64) * 1000u64 + conv ($'!d.subsec_millis()' : u32)

:= rust_last_write_ms (path : string) : u64 =
    := found = rust_metadata_of path
    ? rust_metadata_ok found |> not ?> 0u64
    |:
        := modified = found |> rust_metadata_value |> rust_modified_of
        ? rust_modified_ok modified ?> modified |> rust_modified_value |> rust_since_epoch |> rust_elapsed_ms |: 0u64

:= rust_metadata_len (m : rust_metadata) : u64 = $'!m.len()'

:= rust_symlink_metadata_of (path : string) : rust_metadata_found = $'std::fs::symlink_metadata(&*!path)'

:= rust_metadata_is_symlink (m : rust_metadata) : bool = $'!m.file_type().is_symlink()'

:= rust_file_length (path : string) : i64 =
    := found = rust_metadata_of path
    ? rust_metadata_ok found |> not ?> 0i64 |: found |> rust_metadata_value |> rust_metadata_len |> conv

:= rust_is_symlink (path : string) : bool =
    := found = rust_symlink_metadata_of path
    ? rust_metadata_ok found |> not ?> !- |: found |> rust_metadata_value |> rust_metadata_is_symlink

#### rust_file_length_and_rust_is_symlink_read_one_path (test)

In [39]:
///- --test --symbol
///> rust -d chrono

:= root, disposable = create_temp_directory ()
~~
    := file = root </> "a.txt"
    "abc" |> write_all_text file
    := target = root </> "target"
    target |> create_dir_all |> resultm::unwrap'
    := link = root </> "link"
    directory_create_symbolic_link target link
    := missing = root </> "missing.txt"
    := lengths = rust_file_length file, rust_file_length missing
    := links = rust_is_symlink file, rust_is_symlink link, rust_is_symlink missing
    (lengths, links) |> _assert_eq ((3i64, 0i64), (!-, !+, !-))
    disposable |> use |> ignore

17:43:44 1v file_system.create_dir / { dir = /tmp/!create_temp_path_/spiral_bfbb52ba667053c9773efcc134a9a7b24448b5166dc2d3097b22975ab142b85a/20261107-1743-4410-0639-0000005eb5bf }
__assert_eq { expected = ((3, 0), false, true, false) }

In [ ]:
:= rust_append_opened (path : string) : rust_file_opened = $'std::fs::OpenOptions::new().create(true).append(true).ope\
    n(&*!path)'

:= rust_file_opened_ok (r : rust_file_opened) : bool = $'!r.is_ok()'

:= rust_file_of (r : rust_file_opened) : rust_open_file = $'!r.unwrap()'

:= rust_written_all (f : rust_open_file) (text : string) : rust_io_done = $'std::io::Write::write_all(&mut !f, \
    (!text).as_bytes())'

:= rust_io_ok (r : rust_io_done) : bool = $'!r.is_ok()'

:= rust_append_text (path : string) (text : string) : () =
    := opened = rust_append_opened path
    ? rust_file_opened_ok opened |> not ?> failwith "file_system.append_all_text: the file could not be opened"
    := file = rust_file_of opened
    ? rust_written_all file text |> rust_io_ok |> not ?> failwith "file_system.append_all_text: the write failed"

:~ last_write_ms (path : string) : u64 =
    run_target_args' path \|
        | Fsharp => \ path =>
            ? $'System.IO.File.Exists !path' : bool
            ?> $'uint64 ((System.IO.File.GetLastWriteTimeUtc !path) - System.DateTime.UnixEpoch).TotalMilliseconds' : u64
            |: 0u64
        | Rust => \ path =>
            rust_last_write_ms path
        | TypeScript => \ path =>
            global "import * as spiral_node_fs from 'node:fs';"
            ? $'spiral_node_fs.existsSync(!path)' : bool
            ?> $'BigInt(Math.trunc(spiral_node_fs.statSync(!path).mtimeMs))' : u64
            |: 0u64
        | Gleam => \ path =>
            file_system_gleam_stat ()
            gleam_mtime_ms path
        | _ => \ _ => null ()

#### last_write_ms_is_zero_for_a_missing_path_and_positive_for_a_written_file (test)

In [40]:
///- --test --symbol
///> rust

:= missing = get_temp_path () </> "spiral-last-write-missing.txt"
:= probe = get_temp_path () </> "spiral-last-write-probe.txt"
file_delete missing
last_write_ms missing |> _assert_eq 0u64
write_all_text probe "x"
(last_write_ms probe > 0u64) |> _assert_eq !+
file_delete probe

__assert_eq { expected = 0 }
__assert_eq { expected = true }

0

In [ ]:
:= append_all_text (path : string) (text : string) : () =
    run_target_args' (path, text) \|
        | Fsharp => \ path, text =>
            $'System.IO.File.AppendAllText (!path, !text)'
        | Rust => \ path, text =>
            rust_append_text path text
        | Gleam => \ path, text =>
            file_system_gleam_paths ()
            gleam_append_text path text
        | Lua => \ path, text =>
            := file = lua_file_open path "a"
            lua_file_write file text
            lua_file_close file
        | _ => \ _ => ()

:= appended_text () : string =
    := probe = get_temp_path () </> "spiral-append-probe.txt"
    file_delete probe
    append_all_text probe "a"
    append_all_text probe "b"
    := text = read_all_text probe
    file_delete probe
    text

#### append_all_text_adds_to_the_end_of_a_file (test)

In [41]:
///- --test --symbol
///> rust

appended_text () |> _assert_eq "ab"

__assert_eq { expected = ab }

0

#### lua_files_write_read_append_and_delete (test)

In [42]:
///- --test --symbol
///> lua

:= path = "spiral-lua-file-probe.txt"
file_delete path
path |> file_exists |> _assert_eq !-
write_all_text path "ab"
path |> file_exists |> _assert_eq !+
append_all_text path "c"
path |> read_all_text |> _assert_eq "abc"
file_delete path
path |> file_exists |> _assert_eq !-

__assert_eq { expected = false }
__assert_eq { expected = true }
__assert_eq { expected = abc }
__assert_eq { expected = false }

In [ ]:
:= rust_try_append_text (path : string) (text : string) : result () string =
    := opened = rust_append_opened path
    ? rust_file_opened_ok opened |> not ?> Error "the file could not be opened"
    |:
        := file = rust_file_of opened
        ? rust_written_all file text |> rust_io_ok ?> Ok () |: Error "the write failed"

:= try_append_all_text (path : string) (text : string) : result () string =
    run_target_args' (path, text) \|
        | Rust => \ path, text => rust_try_append_text path text
        | _ => \ path, text =>
            append_all_text path text
            Ok ()

#### try_append_all_text_reports_instead_of_failing (test)

In [43]:
///- --test --symbol
///> rust

:= missing = get_temp_path () </> "spiral-no-such-folder-for-append" </> "x.txt"
(try_append_all_text missing "a" = Error "the file could not be opened") |> _assert_eq !+

__assert_eq { expected = true }

0

In [44]:
:= try_write_all_text (path : string) (text : string) : result () string =
    run_target_args' (path, text) \|
        | Rust => \ path, text =>
            ? (!\\((path, text), $'"std::fs::write(&*$0, &*$1).is_ok()"') : bool) ?> Ok () |: Error "the write failed"
        | _ => \ path, text =>
            write_all_text path text
            Ok ()

#### try_write_all_text_reports_instead_of_failing (test)

In [45]:
///- --test --symbol
///> rust

:= missing = get_temp_path () </> "spiral-no-such-folder-for-write" </> "x.txt"
(try_write_all_text missing "a" = Error "the write failed") |> _assert_eq !+
:= path = get_temp_path () </> "spiral-try-write-all-text.txt"
(try_write_all_text path "ab" = Ok ()) |> _assert_eq !+
path |> read_all_text |> _assert_eq "ab"
file_delete path

__assert_eq { expected = true }
__assert_eq { expected = true }
__assert_eq { expected = ab }

0